# **SVD Model**

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import time
from IPython.display import display
from tqdm.notebook import tqdm
from pathlib import Path
from surprise import Reader, Dataset, SVD
from surprise.model_selection import GridSearchCV
from surprise import accuracy
from sklearn.metrics import ndcg_score

### **Load Data and Train-Test Split**

##### Load Data

In [2]:
# Download data if it isn't downloaded already

if not Path("books_db/ratings.csv").exists():
    kagglehub.dataset_download("zygmunt/goodbooks-10k", output_dir="books_db")
else:
    print("File already exists")

File already exists


In [3]:
# Read in csvs
file_path = "books_db"
ratings_df = pd.read_csv(f"{file_path}/ratings.csv")
books_df = pd.read_csv(f"{file_path}/books.csv")
display(ratings_df.head())
display(ratings_df.shape)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


(981756, 3)

##### Train-Test Split

In [4]:
# This split is kept consistent across models (SVD in this case doesn't need validation so I combine train and val)
rating_counts = ratings_df['user_id'].value_counts()
eligible_users = rating_counts[rating_counts >= 10].index
ratings_eligible = ratings_df[ratings_df['user_id'].isin(eligible_users)].copy()

def split_group_three_way(group, train_frac=0.6, val_frac=0.2, seed=42):
    train = group.sample(frac=train_frac, random_state=seed)
    remaining = group.drop(train.index)
    val = remaining.sample(frac=val_frac / (1 - train_frac), random_state=seed)
    test = remaining.drop(val.index)
    return train, val, test

train_parts = []
val_parts = []
test_parts = []

for user_id, group in ratings_eligible.groupby('user_id'):
    train, val, test = split_group_three_way(group)
    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

ratings_train = pd.concat(train_parts).reset_index(drop=True)
ratings_val = pd.concat(val_parts).reset_index(drop=True)
ratings_test = pd.concat(test_parts).reset_index(drop=True)

In [5]:
train_data = pd.concat([ratings_train, ratings_val])
train_data

,book_id,user_id,rating
0,1199,7,4
1,3246,7,4
2,1646,7,3
3,585,7,4
4,4459,7,4
...,...,...,...
170705,7667,53422,4
170706,4071,53422,4
170707,4483,53424,5
170708,5301,53424,5


In [6]:
test_data = ratings_test
test_data

,book_id,user_id,rating
0,956,7,5
1,1801,7,5
2,1923,7,4
3,2189,7,3
4,2325,7,4
...,...,...,...
171970,5811,53422,4
171971,8757,53422,5
171972,7212,53424,4
171973,7503,53424,4


### **Pre-Processing**

In [7]:
# get the total pool of books and check min and max ID
all_unique_books = pd.concat([books_df['book_id'], ratings_df['book_id']]).unique()
all_books_pool = np.array(all_unique_books)
print(f"Total unique books in the pool: {len(all_books_pool)}")

print("Contains NaN?", pd.isna(all_books_pool).any())

print("Min ID:", np.nanmin(all_books_pool))
print("Max ID:", np.nanmax(all_books_pool))
print("Total unique books in the pool:", len(all_books_pool))

Total unique books in the pool: 19188
Contains NaN? False
Min ID: 1
Max ID: 33288638
Total unique books in the pool: 19188


In [8]:
user_interacted_items = ratings_df.groupby('user_id')['book_id'].apply(set).to_dict()

### **Create Model**

In [9]:
# Give surprise the ratings scale
reader = Reader(rating_scale=(1,5))

# Load data into surprise
dataset = Dataset.load_from_df(train_data[['user_id', 'book_id', 'rating']], reader)
trainset = dataset.build_full_trainset()

# Create parameters for grid search (tested others before this)
param_grid = {
    'n_factors': [200, 250],
    'n_epochs': [50, 75],
    'lr_all': [0.02, 0.03],
    'reg_all': [0.1, 0.2]
}

# Set up grid search
grid_search = GridSearchCV(SVD, param_grid, measures=['rmse', 'fcp'], cv=3, n_jobs=-1)

# Fit grid search to dataset and record time taken
print("Grid search is running...")
start_time = time.time()
grid_search.fit(dataset)
end_time = time.time()
print(f"Grid Serach Time Taken: {(end_time - start_time):.2f} seconds")

# Print best params (rmse and fcp, fcp because final evaluation will be based on ranking)
print(f"Best RMSE Parameters: {grid_search.best_params['rmse']}")
print(f"Best RMSE Score: {grid_search.best_score['rmse']:.5f}")
print(f"Best FCP Parameters: {grid_search.best_params['fcp']}")
print(f"Best FCP Score: {grid_search.best_score['fcp']:.5f}")

Grid search is running...
Grid Serach Time Taken: 200.62 seconds
Best RMSE Parameters: {'n_factors': 250, 'n_epochs': 75, 'lr_all': 0.02, 'reg_all': 0.1}
Best RMSE Score: 0.82619
Best FCP Parameters: {'n_factors': 250, 'n_epochs': 50, 'lr_all': 0.03, 'reg_all': 0.1}
Best FCP Score: 0.62432


In [10]:
# Choosing fcp because the evaluation across all models will be based on ranking
optimal_params = grid_search.best_params['fcp']

# Train Model
svd = SVD(**optimal_params)
svd.fit(trainset)

### **Evaluation**

##### RMSE, MAE

In [11]:
test_data_formatted = list(zip(test_data['user_id'], test_data['book_id'], test_data['rating']))
svd_predictions = svd.test(test_data_formatted)
accuracy.rmse(svd_predictions)
accuracy.mae(svd_predictions)

RMSE: 0.8064
MAE:  0.6237


np.float64(0.6236559251261575)

##### NDCG@K

In [12]:
k = 5
num_negatives = 100
user_ndcg_scores = []

for user, group in test_data.groupby('user_id'):
    test_true_ratings = group['rating'].values

    # Predictions for books actually rated
    test_pred_ratings = np.array([svd.predict(user, book).est for book in group['book_id']])

    # Get pool of never rated items
    interacted = user_interacted_items.get(user, set())
    valid_negatives = np.setdiff1d(all_books_pool, list(interacted))

    # Get the decoys
    samples_negatives = np.random.choice(valid_negatives, size=num_negatives, replace=False)

    # Predict ratings for the decoys
    neg_pred_ratings = np.array([svd.predict(user, neg_book).est for neg_book in samples_negatives])

    # combine actual books and decoys (true ratings of 0 assigned to decoys)
    combined_true = np.concatenate([test_true_ratings, np.zeros(num_negatives)])
    combined_pred = np.concatenate([test_pred_ratings, neg_pred_ratings])

    y_true = combined_true.reshape(1, -1)
    y_score = combined_pred.reshape(1, -1)

    score = ndcg_score(y_true, y_score, k=k)
    user_ndcg_scores.append(score)

mean_ndcg = np.mean(user_ndcg_scores)

print(f"NDCG@{k}: {mean_ndcg:.4f}")

NDCG@5: 0.1507
